# Phase 1 check — retrieval vs. gold citations

Runs **Phase 1 only** (PERCEIVE → FORMULATE → RETRIEVE ↔ ASSESS, via
`agent.pipeline.run_phase1`) for a handful of items and prints a step-by-step
trace per item:

1. **PERCEIVE** — focus / vehicle / observed / violation / confidence.
2. **Query per round** — the query text(s) actually issued on each
   refine-loop round, in order (round 0 from `formulate()`, later rounds
   rebuilt from ASSESS's `refined_query`).
3. **Final retrieved vs. gold** — the final retrieved chunk ids (score-sorted
   — same order SELECT would see) against the item's **gold** chunk ids
   (`law` field in `law.json`, resolved through `CitationIndex`).

No SELECT call, no answer — but PERCEIVE, ASSESS, and FORMULATE's query text
all see the MCQ options (AGENT_PLAN_v2.md §1): PERCEIVE/ASSESS use them only
to steer attention/judge sufficiency, never to answer; FORMULATE appends the
option text onto every retrieval query it builds, on top of the
perception-grounded violation phrase. As of v2.7, every query — the first
round from `formulate()` and every refine-loop round rebuilt from ASSESS's
`refined_query` — states the vehicle explicitly and matches its consequence
phrase to what the question actually asks (mức phạt / trừ điểm / tước
quyền), never falling back to scene-register text (§3.2). The "Test
FORMULATE" cell below shows exactly what that query text looks like, with
and without options, for whichever item the single-item deep dive above just
ran.

**Kernel**: run this under the `traffic` conda env kernel — it's the one
with pyserini + the bundled Java 21 JDK (needed for BM25) and the matching
`torch`/`torchvision` cu128 build (needed for `--llm vlm`) already set up.
If you only want to smoke-test the plumbing without a GPU, set
`LLM_BACKEND = "mock"` below and skip the VLM load entirely.


In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "agent").exists():
    ROOT = ROOT.parent  # running from notebooks/, repo root is one level up
assert (ROOT / "agent").exists(), f"can't find the agent/ package from {ROOT}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from PIL import Image

from agent.cli import resolve_image_path
from agent.corpus import Corpus
from agent.pipeline import run_phase1
from agent.resolve import CitationIndex
from agent.retrieval.bm25 import DEFAULT_INDEX_DIR, BM25Retriever
from agent.retrieval.mock import MockRetriever



In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2" # avoid warning from huggingface tokenizers

## Config — edit this cell, then run everything below

In [ ]:
# --- data ---
DATA_PATH = ROOT / "law.json"
CORPUS_PATH = ROOT / "law_db.json"
DATA_ROOT = ROOT

# --- retriever: "bm25" (real, needs the Lucene index built once via
#     `python -m agent.build_bm25_index`) or "mock" (deterministic, no index) ---
RETRIEVER = "bm25"
INDEX_DIR = DEFAULT_INDEX_DIR
BM25_K1, BM25_B = 0.9, 0.4

# --- llm: "mock" (deterministic, zero-download, for plumbing checks) or
#     "vlm" (real model via answer_vqa.PRESETS — needs a GPU node) ---
LLM_BACKEND = "vlm"
MODEL_PRESET = "qwen3vl-4b"  # see answer_vqa.PRESETS for the full list
DEVICE = "cuda"
DTYPE = "bfloat16"

# --- phase 1 knobs (mirrors agent.cli defaults) ---
K = 3                  # chunks retrieved per query, per round
MAX_ITERATIONS = 2     # RETRIEVE rounds; ASSESS runs on all but the last
AS_OF = None           # effective_date filter, e.g. "2025-01-01"

# --- how many items to look at ---
N_SAMPLES = 20          # batch evaluation sample size
ITEM_INDEX = 0          # index into `records` for the single-item deep dive


In [ ]:
records = json.loads(DATA_PATH.read_text(encoding="utf-8"))
corpus = Corpus.load(CORPUS_PATH)
citation_index = CitationIndex.from_file(CORPUS_PATH)
print(f"{len(records)} questions | corpus: {len(corpus)} chunks (v{corpus.version})")


In [ ]:
if RETRIEVER == "bm25":
    retriever = BM25Retriever(index_dir=INDEX_DIR, k1=BM25_K1, b=BM25_B)
else:
    retriever = MockRetriever(corpus)
print(f"retriever: {RETRIEVER}")


In [ ]:
if LLM_BACKEND == "vlm":
    from agent.llm_vlm import VLMLLMClient  # heavy: pulls in torch/transformers
    import answer_vqa

    preset = answer_vqa.PRESETS[MODEL_PRESET]
    llm = VLMLLMClient(preset.hf_id, preset.family, device=DEVICE, dtype=DTYPE)
    model_label = MODEL_PRESET
else:
    from agent.llm import MockLLM

    llm = MockLLM()
    model_label = "mock"
print(f"llm: {LLM_BACKEND} ({model_label})")


## Helper: run Phase 1 on one item and score retrieval against gold

In [ ]:
def run_and_compare(rec: dict) -> dict:
    image_path = resolve_image_path(rec["image"], DATA_ROOT)
    gold_resolved = citation_index.resolve_set(rec.get("law", []))
    options = {L: rec[L] for L in ("A", "B", "C", "D")}

    perception, retrieved, trace = run_phase1(
        image_path, rec["question"], options, corpus, retriever, llm,
        k=K, as_of=AS_OF, max_iterations=MAX_ITERATIONS,
    )
    retrieved_ids = {c.chunk_id for c in retrieved}
    overlap = retrieved_ids & gold_resolved

    n_gold = len(gold_resolved)
    recall = len(overlap) / n_gold if n_gold else None
    precision = len(overlap) / len(retrieved_ids) if retrieved_ids else None

    return {
        "item_id": rec.get("id"),
        "question": rec["question"],
        "image_path": image_path,
        "type_image": rec.get("type_image"),
        "perception": perception,
        "trace": trace,
        "retrieved": retrieved,
        "retrieved_ids": retrieved_ids,
        "gold_ids": gold_resolved,
        "overlap_ids": overlap,
        "n_gold": n_gold,
        "n_retrieved": len(retrieved_ids),
        "n_overlap": len(overlap),
        "recall": recall,
        "precision": precision,
        "iterations_used": trace.iterations_used,
        "stop_reason": trace.stop_reason,
    }


def print_result(res: dict) -> None:
    """Step-by-step trace through Phase 1:
    1. PERCEIVE's output.
    2. Each refine-loop round's query text(s), in order.
    3. Final retrieved chunk ids (score-sorted — same order SELECT would
       see) vs. gold.
    """
    print(f"item_id={res['item_id']}\n")

    p = res["perception"]
    print("1. PERCEIVE:")
    print(f"   focus:      {p.focus}")
    print(f"   vehicle:    {p.vehicle}")
    print(f"   observed:   {p.observed}")
    print(f"   violation:  {p.violation}")
    print(f"   confidence: {p.confidence}\n")

    print(f"2. Queries per round ({res['iterations_used']} round(s), stop_reason={res['stop_reason']}):")
    for it in res["trace"].iterations:
        print(f"   round {it.iteration}: {it.query_texts}")
    print()

    final_chunks = sorted(res["retrieved"], key=lambda c: c.score, reverse=True)
    retrieved_ids_sorted = [c.chunk_id for c in final_chunks]
    print("3. Final retrieved vs. gold:")
    print(f"   retrieved ({len(retrieved_ids_sorted)}): {retrieved_ids_sorted}")
    print(f"   gold      ({res['n_gold']}): {sorted(res['gold_ids'])}")


## Single-item deep dive

In [ ]:
ITEM_INDEX = 3

In [ ]:
rec = records[ITEM_INDEX]

print(f"item_id={rec.get('id')}  type_image={rec.get('type_image')}")
print(f"Q: {rec['question']}\n")
options = {L: rec[L] for L in ("A", "B", "C", "D")}
for L, text in options.items():
    marker = " <- GT" if rec.get("GT") == L else ""
    print(f"{L}. {text}{marker}")
print()

result = run_and_compare(rec)

display(Image.open(result["image_path"]))
print_result(result)


## Test FORMULATE — inspect the actual retrieval query

Reuses the `Perception` PERCEIVE already produced above (no extra model
call) and calls `formulate()` directly to show exactly what query text,
`vehicle_filter`, and `doc_boost` each RETRIEVE call gets.

- **WITH options** — what `e4_agent` actually sends to BM25.
- **WITHOUT options** — same vehicle/focus-phrase logic, just no option
  signal appended (isolates that effect from everything else).

Four things worth checking directly in the output below: (1) `vehicle`
should appear as a natural-language phrase (e.g. "xe mô tô") at the front of
*every* query, not just the penalty one; (2) the second query's suffix
should match what the question actually asks — "trừ điểm giấy phép lái xe"
for a point-deduction question, "tước quyền sử dụng giấy phép lái xe" for a
license-suspension question, "mức phạt tiền" otherwise; (3) if
`perception.violation` is `None`, the query should use `perception.focus`,
never `perception.observed` (scene-register text that can leak non-statute
descriptive noise in); (4) `option_signal` (v2.8) should be just the
number+unit fragments extracted from the options ("02 điểm", "01 tháng đến
03 tháng"), NOT the full option sentences — unless none of the options
contain a number, in which case it falls back to the full text.

The last block below demonstrates the same thing for a **refine-loop round**
(v2.7) — `_build_query_text()` is the exact helper `run_phase1()` uses to
rebuild the query from ASSESS's `refined_query`, so a round-2+ query gets
the same vehicle/focus-phrase/option-signal treatment as round 0, not just a
bare `refined_query` string.

In [ ]:
from agent.pipeline import _build_query_text, _extract_option_signal, _penalty_focus_phrase, formulate, is_penalty_question

test_perception = result["perception"]
test_options = {L: rec[L] for L in ("A", "B", "C", "D")}


def show_queries(label: str, queries) -> None:
    print(f"{label} ({len(queries)} quer{'y' if len(queries) == 1 else 'ies'}):")
    for q in queries:
        print(f"  text:           {q.text}")
        print(f"  vehicle_filter: {q.vehicle_filter}")
        print(f"  doc_boost:      {q.doc_boost}\n")


print(f"perception.violation: {test_perception.violation!r}")
print(f"perception.focus:     {test_perception.focus!r}")
print(f"perception.vehicle:   {test_perception.vehicle!r}")
print(f"penalty_asked:        {is_penalty_question(rec['question'])}")
print(f"focus_phrase:         {_penalty_focus_phrase(rec['question'])!r}")
print(f"option_signal:        {_extract_option_signal(test_options)!r}\n")

show_queries("WITH options (e4_agent behavior)", formulate(test_perception, rec["question"], test_options))
show_queries("WITHOUT options (isolates the options-leakage effect)", formulate(test_perception, rec["question"], None))

# v2.7: a refine-loop round rebuilds its query with the same helper formulate()
# uses, from whatever ASSESS's refined_query happened to be — shown here with
# a made-up refined_query text to confirm the vehicle/focus-phrase/option-signal
# treatment survives past round 0.
vehicle_clause = f"{test_perception.vehicle} " if test_perception.vehicle else ""
penalty_focus = _penalty_focus_phrase(rec["question"]) if is_penalty_question(rec["question"]) else None
option_signal = _extract_option_signal(test_options)
fake_refined_query = "không chấp hành hiệu lệnh của người điều khiển giao thông"
print(f"Simulated refine-loop round (refined_query={fake_refined_query!r}):")
print(f"  text: {_build_query_text(fake_refined_query, vehicle_clause, option_signal, penalty_focus)}")
